# Bi-2223 Phase 1 baseline surrogates

Load and compare all trained Bi-2223 surrogates against the analytic ground truth
(Marchevsky & Prestemon 2024). Three input-based (A, B, raw) and two sequence-based
(GRU, Transformer) models.

**Expects trained model files** — run `scripts/train_eval_bi2223_surrogates.py` first,
or point `MODEL_DIR` at the Eagle output directory.

In [ ]:
import sys, pathlib, os, json
repo_root = pathlib.Path.cwd()
if not (repo_root / "hts_tape_env").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Point this at wherever the trained models live
MODEL_DIR = repo_root / "models"
# MODEL_DIR = pathlib.Path("/eagle/fanc_ldrd/magnets_fem/bi2223_baselines")
print("repo root:", repo_root)
print("model dir:", MODEL_DIR)

## 1. Load trained models

In [ ]:
from hts_tape_env.surrogates import load_surrogate
from hts_tape_env.seq_surrogates import load_seq_surrogate

# Input-based surrogates
input_models = {}
for spec in ("A", "B", "raw"):
    p = MODEL_DIR / f"bi2223_surr_{spec}.joblib"
    if p.exists():
        input_models[spec] = load_surrogate(p)
        print(f"Loaded {spec}: {p.name}")
    else:
        print(f"  {spec}: not found at {p}")

# Sequence surrogates
seq_models = {}
for arch in ("gru", "transformer"):
    p = MODEL_DIR / f"bi2223_seq_{arch}.pt"
    if p.exists():
        seq_models[arch] = load_seq_surrogate(p)
        print(f"Loaded {arch}: {p.name}")
    else:
        print(f"  {arch}: not found at {p}")

# Metrics JSON
metrics_path = MODEL_DIR / "bi2223_eval_metrics.json"
if metrics_path.exists():
    with open(metrics_path) as f:
        all_metrics = json.load(f)
    print(f"\nLoaded metrics from {metrics_path.name}")
else:
    all_metrics = None
    print("  metrics JSON not found — will compute from scratch")

## 2. Input-based surrogate metrics

Specs A (T0 → I\*), B (T0, I → I\*), raw (T0, I, V, T\_surface → I\*).

In [ ]:
if all_metrics:
    rows = []
    for spec in ("A", "B", "raw"):
        key = f"input_{spec}"
        if key in all_metrics:
            m = all_metrics[key]
            rows.append({"model": f"input ({spec})",
                         "MAE [A]": f"{m['mae']:.2f}",
                         "R²": f"{m['r2']:.4f}",
                         "median rel err": f"{m['median_rel_err']:.2%}"})
    display(pd.DataFrame(rows))
else:
    print("No saved metrics — run the training script first.")

## 3. Sequence surrogate metrics

Five-head GRU / Transformer: margin, breach logit, log I\*, T\*, log time margin.

In [ ]:
if all_metrics:
    rows = []
    for arch in ("gru", "transformer"):
        key = f"seq_{arch}"
        if key in all_metrics:
            m = all_metrics[key]
            rows.append({
                "arch": arch,
                "best val loss": f"{m['best_val_loss']:.4f}",
                "I* MAE [A]": f"{m['istar_mae_A']:.2f}",
                "I* med rel err": f"{m['istar_median_rel_err']:.2%}",
                "T* MAE [K]": f"{m['tstar_mae_K']:.3f}",
                "breach acc": f"{m['breach_accuracy']:.2%}",
                "tm med rel err": f"{m.get('time_margin_median_rel_err', float('nan')):.2%}",
            })
    display(pd.DataFrame(rows))
else:
    for arch, sr in seq_models.items():
        print(f"{arch}: {sr.metrics}")

## 4. I\*(T0) comparison — analytic vs all surrogates

In [ ]:
from hts_tape_env.physics_bi2223 import solve_runaway, T_ANCHOR_K
from hts_tape_env.thermal_bi2223 import time_margin_to_safe
from hts_tape_env.bi2223_rollout import generate_bi2223_grid_dataset, rollout_bi2223_episode
from hts_tape_env.seq_surrogates import predict_episode
from hts_tape_env.seq_data import TRUE_FEATURES

T0_LO, T0_HI, N_EVAL = 77.16, 80.5, 30
T0_eval = np.linspace(T0_LO, T0_HI, N_EVAL)

# Analytic ground truth
Istar_analytic = np.full(N_EVAL, np.nan)
Tstar_analytic = np.full(N_EVAL, np.nan)
tm_analytic_ms = np.full(N_EVAL, np.nan)
for i, T0 in enumerate(T0_eval):
    Is, Ts = solve_runaway(T0)
    if np.isfinite(Is) and Is > 0:
        Istar_analytic[i] = Is
        Tstar_analytic[i] = Ts
        tm_s = time_margin_to_safe(Is, Ts)
        if np.isfinite(tm_s):
            tm_analytic_ms[i] = tm_s * 1000.0

# Input-based: spec A
Istar_input_A = np.full(N_EVAL, np.nan)
if "A" in input_models:
    Istar_input_A = input_models["A"]["model"].predict(T0_eval.reshape(-1, 1))

# Input-based: spec raw
Istar_input_raw = np.full(N_EVAL, np.nan)
if "raw" in input_models:
    for i, T0 in enumerate(T0_eval):
        ep = rollout_bi2223_episode(T0, episode_id=0)
        if not ep:
            continue
        mid = len(ep) // 2
        r = ep[mid]
        X_raw = np.array([[r["T0_K"], r["I_true_A"], r["voltage_true_V"], r["T_surface_true_K"]]])
        Istar_input_raw[i] = input_models["raw"]["model"].predict(X_raw)[0]

# Sequence surrogates
df_eval = generate_bi2223_grid_dataset(T0_eval)
seq_Istar = {}
seq_Tstar = {}
seq_tm_ms = {}
trajectory_data = {}

for arch, sr in seq_models.items():
    Is = np.full(N_EVAL, np.nan)
    Ts = np.full(N_EVAL, np.nan)
    tm = np.full(N_EVAL, np.nan)
    for _eid, g in df_eval.sort_values("step").groupby("episode_id", sort=True):
        T0 = g["T0_K"].iloc[0]
        X = g[list(TRUE_FEATURES)].to_numpy(dtype=np.float32)
        r = predict_episode(sr, X)
        idx = np.argmin(np.abs(T0_eval - T0))
        Is[idx] = r["istar_pred_A"]
        Ts[idx] = r["tstar_pred_K"]
        tm[idx] = r["time_margin_pred_ms"]
        if arch == list(seq_models.keys())[0]:
            trajectory_data[T0] = g
    seq_Istar[arch] = Is
    seq_Tstar[arch] = Ts
    seq_tm_ms[arch] = tm

print(f"Evaluation grid: {N_EVAL} points from {T0_LO} to {T0_HI} K")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
colors = {"gru": "tab:blue", "transformer": "tab:orange"}
markers = {"gru": "^", "transformer": "v"}

# I*(T0)
ax = axes[0]
ax.plot(T0_eval, Istar_analytic, "k-", lw=2, label="analytic")
ax.plot(T0_eval, Istar_input_A, "s", ms=5, color="tab:green", label="input (A)")
for arch, Is in seq_Istar.items():
    ax.plot(T0_eval, Is, markers[arch], ms=5, color=colors[arch], label=f"seq ({arch})")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("I* [A]")
ax.set_title("Runaway current I*(T0)"); ax.legend(fontsize=8)

# T*(T0)
ax = axes[1]
ax.plot(T0_eval, Tstar_analytic, "k-", lw=2, label="analytic")
for arch, Ts in seq_Tstar.items():
    ax.plot(T0_eval, Ts, markers[arch], ms=5, color=colors[arch], label=f"seq ({arch})")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("T* [K]")
ax.set_title("Runaway temperature T*(T0)"); ax.legend(fontsize=8)

# Time margin
ax = axes[2]
valid_a = np.isfinite(tm_analytic_ms)
ax.plot(T0_eval[valid_a], tm_analytic_ms[valid_a], "k-", lw=2, label="analytic")
for arch, tm in seq_tm_ms.items():
    valid_s = np.isfinite(tm)
    if valid_s.any():
        ax.plot(T0_eval[valid_s], tm[valid_s], markers[arch], ms=5,
                color=colors[arch], label=f"seq ({arch})")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("Time margin [ms]")
ax.set_title("Adiabatic time margin at runaway"); ax.legend(fontsize=8)

fig.suptitle("Bi-2223 thermal runaway surface — analytic vs surrogates", fontsize=13)
fig.tight_layout()
plt.show()

## 5. Error analysis — relative error vs T0

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
denom = np.maximum(np.abs(Istar_analytic), 1e-6)

# Relative error
ax = axes[0]
rel_err_A = np.abs(Istar_input_A - Istar_analytic) / denom
rel_err_raw = np.abs(Istar_input_raw - Istar_analytic) / denom
ax.plot(T0_eval, rel_err_A * 100, "s-", ms=4, color="tab:green", label="input A")
ax.plot(T0_eval, rel_err_raw * 100, "d-", ms=4, color="tab:purple", label="input raw")
for arch, Is in seq_Istar.items():
    rel_err = np.abs(Is - Istar_analytic) / denom
    ax.plot(T0_eval, rel_err * 100, f"{markers[arch]}-", ms=4,
            color=colors[arch], label=f"seq {arch}")
ax.axvspan(80.0, 80.5, color="gray", alpha=0.2, label="unreliable n(T)")
ax.set_xlabel("T0 [K]"); ax.set_ylabel("I* relative error [%]")
ax.set_title("Surrogate error vs T0"); ax.legend(fontsize=7)

# Residuals near boundary
ax = axes[1]
mask = T0_eval >= 79.5
if mask.any() and np.any(np.isfinite(Istar_analytic[mask])):
    ax.plot(T0_eval[mask], Istar_input_A[mask] - Istar_analytic[mask],
            "s-", ms=5, color="tab:green", label="input A")
    for arch, Is in seq_Istar.items():
        ax.plot(T0_eval[mask], Is[mask] - Istar_analytic[mask],
                f"{markers[arch]}-", ms=5, color=colors[arch], label=f"seq {arch}")
    ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("T0 [K]"); ax.set_ylabel("I* residual [A]")
ax.set_title("Residuals near n(T)→1 boundary (T0 ≥ 79.5 K)")
ax.legend(fontsize=8)

fig.tight_layout()
plt.show()

## 6. Sample trajectories

In [ ]:
sample_T0s = [77.5, 78.5, 79.5, 80.2]
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for idx, T0_target in enumerate(sample_T0s):
    ax = axes[idx // 2, idx % 2]
    if not trajectory_data:
        ax.set_title(f"T0 ≈ {T0_target:.1f} K — no data")
        continue
    T0_key = min(trajectory_data.keys(), key=lambda t: abs(t - T0_target))
    g = trajectory_data[T0_key]
    step = g["step"].to_numpy()

    ax.plot(step, g["I_true_A"], color="tab:blue", lw=1.5, label="I [A]")
    istar_ep = g["Istar_A"].iloc[0]
    ax.axhline(istar_ep, color="gray", ls=":", lw=1, label=f"I*={istar_ep:.0f} A")
    ax.set_xlabel("step"); ax.set_ylabel("I [A]", color="tab:blue")
    ax.set_title(f"T0 = {T0_key:.2f} K", fontsize=10)

    ax2 = ax.twinx()
    ax2.plot(step, g["voltage_true_V"] * 1e6, color="tab:red", ls="--", lw=1.5, label="V [µV]")
    ax2.set_ylabel("V [µV]", color="tab:red")

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7, loc="upper left")

fig.suptitle("Bi-2223 ramp trajectories — I and V vs step", fontsize=12)
fig.tight_layout()
plt.show()

## 7. Summary table — all models side by side

In [ ]:
rows = []

for spec in ("A", "B", "raw"):
    key = f"input_{spec}"
    if all_metrics and key in all_metrics:
        m = all_metrics[key]
        eval_m = all_metrics.get("eval", {}).get(f"input_{spec}", {})
        rows.append({
            "model": f"input ({spec})",
            "type": "input-based",
            "train MAE [A]": f"{m['mae']:.2f}",
            "train R²": f"{m['r2']:.4f}",
            "eval I* med rel err": f"{eval_m.get('istar_median_rel_err', float('nan')):.2%}" if eval_m else "—",
            "eval I* 95th rel err": f"{eval_m.get('istar_95th_rel_err', float('nan')):.2%}" if eval_m else "—",
        })

for arch in ("gru", "transformer"):
    key = f"seq_{arch}"
    if all_metrics and key in all_metrics:
        m = all_metrics[key]
        eval_m = all_metrics.get("eval", {}).get(key, {})
        rows.append({
            "model": f"seq ({arch})",
            "type": "sequence",
            "train MAE [A]": f"{m.get('istar_mae_A', float('nan')):.2f}",
            "train R²": "—",
            "eval I* med rel err": f"{eval_m.get('istar_median_rel_err', float('nan')):.2%}" if eval_m else "—",
            "eval I* 95th rel err": f"{eval_m.get('istar_95th_rel_err', float('nan')):.2%}" if eval_m else "—",
        })

if rows:
    display(pd.DataFrame(rows).set_index("model"))
else:
    print("No metrics available — run training first.")

## Next steps

1. **Phase 2** — First-principles REBCO via FEM: use the FreeFEM 3D-slit models
   (real material properties) to drive REBCO physics instead of the lumped analytic model.
2. **Phase 3** — Pipe FEM data (sweep.h5: 6 episodes, 16 RTDs + 20 voltage taps)
   into surrogate training, not just inference.
3. More evaluation points, especially near the n(T)→1 boundary at T0 ≈ 80.5 K.